# Clase 087 — SHAP en profundidad

TreeExplainer sobre GBM (rápido y exacto) + DeepExplainer/KernelExplainer sobre MLP.

Instalar: `pip install shap`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split

try:
    import shap
    SHAP_OK = True
except ImportError:
    print('instalar: pip install shap')
    SHAP_OK = False

np.random.seed(42)

## 1. Dataset sintético con nombres de features

In [ ]:
feat_names = ['income', 'age', 'score_a', 'score_b', 'tenure', 'debt_ratio',
              'n_products', 'is_active', 'region_enc', 'channel_enc']
X, y = make_classification(n_samples=2000, n_features=10, n_informative=6,
                            n_redundant=2, weights=[0.6, 0.4], random_state=42)
X = pd.DataFrame(X, columns=feat_names)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,
                                                      stratify=y, random_state=42)
print('train', X_train.shape, 'test', X_test.shape)

## 🧠 Intuición previa

Antes de entrar al detalle: **SHAP reparte el "crédito" de una predicción entre las features de
forma justa**, como dividir la cuenta de una cena según lo que pidió cada comensal. Esa idea de
reparto justo viene de los **valores de Shapley** de la teoría de juegos: cada feature recibe la
parte de la predicción que le corresponde por su contribución marginal promediada sobre todos los
órdenes posibles. Por eso la suma de los valores SHAP más el valor base reconstruye exactamente la
predicción del modelo.

## 2. GBM + TreeExplainer

In [ ]:
gbm = GradientBoostingClassifier(n_estimators=200, max_depth=4, random_state=42)
gbm.fit(X_train, y_train)
print(f'GBM acc test: {gbm.score(X_test, y_test):.4f}')

if SHAP_OK:
    explainer = shap.TreeExplainer(gbm)
    shap_values = explainer.shap_values(X_test)
    print('shap_values shape:', np.array(shap_values).shape)
    print('expected_value (baseline):', explainer.expected_value)

## 3. Importancia global (mean |SHAP|)

In [ ]:
if SHAP_OK:
    sv = np.array(shap_values)
    if sv.ndim == 3:  # newer SHAP can return (n, f, classes)
        sv = sv[..., 1]
    global_imp = pd.Series(np.abs(sv).mean(axis=0), index=feat_names).sort_values(ascending=False)
    print(global_imp.round(4).to_string())

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.barh(global_imp.index[::-1], global_imp.values[::-1], color='#37a')
    ax.set_xlabel('mean(|SHAP value|)')
    ax.set_title('Importancia global SHAP')
    plt.tight_layout()
    plt.show()

## 4. Summary plot (beeswarm)

In [ ]:
if SHAP_OK:
    shap.summary_plot(sv, X_test, feature_names=feat_names, show=True, plot_size=(7, 4))

## 5. Dependence plot (no-linearidad + interacciones)

In [ ]:
if SHAP_OK:
    top_feature = global_imp.index[0]
    shap.dependence_plot(top_feature, sv, X_test, feature_names=feat_names, show=True)

## 6. Force plot conceptual (suma de contribuciones para 1 muestra)

In [ ]:
if SHAP_OK:
    idx = 0
    base = float(np.array(explainer.expected_value).ravel()[-1])
    contribs = pd.Series(sv[idx], index=feat_names).sort_values()
    pred = base + contribs.sum()
    print(f'expected_value (baseline log-odds): {base:.4f}')
    print(f'sum SHAP contributions:             {contribs.sum():.4f}')
    print(f'reconstructed prediction:           {pred:.4f}')

    fig, ax = plt.subplots(figsize=(7, 4))
    colors = ['#c33' if v > 0 else '#37a' for v in contribs.values]
    ax.barh(contribs.index, contribs.values, color=colors)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_xlabel('SHAP contribution (signo = dirección)')
    ax.set_title(f'Force plot conceptual — muestra {idx}')
    plt.tight_layout()
    plt.show()

## 7. Interaction values

In [ ]:
if SHAP_OK:
    # Tomamos subset chico (interaction es O(n_features^2 · n_samples))
    inter = explainer.shap_interaction_values(X_test.iloc[:200])
    inter_arr = np.array(inter)
    if inter_arr.ndim == 4:
        inter_arr = inter_arr[..., 1]
    # mean |interaction| por par
    pair_imp = np.abs(inter_arr).mean(axis=0)
    np.fill_diagonal(pair_imp, 0)
    i, j = np.unravel_index(np.argmax(pair_imp), pair_imp.shape)
    print(f'Top interaction: {feat_names[i]} <-> {feat_names[j]} = {pair_imp[i, j]:.4f}')

## 8. MLP + DeepExplainer (con fallback a KernelExplainer)

DeepExplainer espera modelos Keras/PyTorch — sobre `sklearn.MLPClassifier` cae a KernelExplainer model-agnostic.

In [ ]:
mlp = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=300, random_state=42)
mlp.fit(X_train, y_train)
print(f'MLP acc test: {mlp.score(X_test, y_test):.4f}')

if SHAP_OK:
    explainer_mlp = None
    try:
        explainer_mlp = shap.DeepExplainer(mlp, X_train.values[:100])
        sv_mlp = explainer_mlp.shap_values(X_test.values[:50])
        print('DeepExplainer ok')
    except Exception as e:
        print(f'DeepExplainer no aplica ({type(e).__name__}); fallback KernelExplainer.')
        background = shap.sample(X_train, 50, random_state=42)
        explainer_mlp = shap.KernelExplainer(mlp.predict_proba, background)
        sv_mlp = explainer_mlp.shap_values(X_test.iloc[:30], nsamples=100, silent=True)
        sv_mlp_pos = np.array(sv_mlp)
        if sv_mlp_pos.ndim == 3:
            sv_mlp_pos = sv_mlp_pos[..., 1] if sv_mlp_pos.shape[-1] == 2 else sv_mlp_pos[-1]
        elif isinstance(sv_mlp, list):
            sv_mlp_pos = np.array(sv_mlp[1])
        print('KernelExplainer shap_values shape:', sv_mlp_pos.shape)
        imp_mlp = pd.Series(np.abs(sv_mlp_pos).mean(axis=0), index=feat_names).sort_values(ascending=False)
        print('\nTop-5 features (MLP):')
        print(imp_mlp.head().round(4).to_string())

## Ejercicios

1. Cambiá a XGBoost. ¿Coincide el top-3 de features con el GBM?
2. Explicá una muestra clasificada erróneamente. ¿Qué feature "empujó" mal?
3. Agrupá features correlacionadas (`shap.utils.hclust`) y rehacé el summary plot.

## Conclusiones

- TreeExplainer es exacto y rápido para tree-based: usalo siempre que puedas.
- KernelExplainer es lento pero model-agnostic — buen fallback.
- SHAP es atribución estadística, **no causal** — features correlacionadas comparten crédito de forma arbitraria.

## ✅ Soluciones de los ejercicios

Cinco ejercicios del README sobre **XGBoost + California Housing**. Como el paquete `shap` puede no estar instalado, usamos un ayudante `shap_de(model, X)` que prefiere `shap` y, si no está, cae en los valores **TreeSHAP nativos de XGBoost** (`pred_contribs` / `pred_interactions`): son los mismos valores de Shapley, sin dependencias extra y sin internet. Submuestreamos para correr en < 60s con `n_jobs=1`.

**Setup — ayudante SHAP con *fallback*.** `pred_contribs=True` de XGBoost devuelve una matriz `(n, n_features+1)` donde la última columna es el valor base; el resto son las contribuciones de Shapley de cada feature.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

try:
    import shap
    SHAP_OK = True
except Exception:
    SHAP_OK = False
print('paquete shap disponible:', SHAP_OK)

data = fetch_california_housing()
rng = np.random.default_rng(42)
idx = rng.choice(len(data.data), 1500, replace=False)
X, y, feat = data.data[idx], data.target[idx], list(data.feature_names)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42)

model = xgb.XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.1,
                         n_jobs=1, random_state=42).fit(Xtr, ytr)

def shap_de(model, Xmat):
    '''Devuelve (shap_values (n,f), base_value). Usa shap si esta; si no, TreeSHAP de XGBoost.'''
    if SHAP_OK:
        expl = shap.TreeExplainer(model)
        sv = np.asarray(expl.shap_values(Xmat))
        base = np.asarray(expl.expected_value).ravel()[0]
        return sv, base
    booster = model.get_booster()
    contribs = booster.predict(xgb.DMatrix(Xmat), pred_contribs=True)
    return contribs[:, :-1], float(contribs[0, -1])

print(f'R2 test XGBoost: {model.score(Xte, yte):.4f}')

**Ejercicio 1 — TreeExplainer.** Calculamos los valores SHAP del test set.

In [ ]:
shap_values, base_value = shap_de(model, Xte)
print('shap_values shape:', shap_values.shape)
print('valor base (E[f(x)]):', round(base_value, 4))
assert shap_values.shape == (len(Xte), len(feat))
print('OK: una contribucion por feature y por instancia')

**Ejercicio 2 — Summary plot.** Importancia global = `mean(|SHAP|)`. Identificamos las 3 features más importantes y su dirección (correlación entre valor de feature y su SHAP).

In [ ]:
imp = np.abs(shap_values).mean(0)
order = np.argsort(imp)
plt.figure(figsize=(7, 4))
plt.barh(np.array(feat)[order], imp[order], color='#37a')
plt.xlabel('mean(|SHAP value|)'); plt.title('Importancia global SHAP')
plt.tight_layout(); plt.show()

top3 = order[::-1][:3]
for j in top3:
    signo = np.corrcoef(Xte[:, j], shap_values[:, j])[0, 1]
    dire = 'sube la prediccion al crecer' if signo > 0 else 'baja la prediccion al crecer'
    print(f'  {feat[j]:12} imp={imp[j]:.3f}  -> {dire}')
print('MedInc suele dominar: mas ingreso mediano -> mayor valor de vivienda.')

**Ejercicio 3 — Waterfall.** Para una muestra, la suma `base + Σ SHAP` reconstruye exactamente la predicción del modelo.

In [ ]:
i = 0
contrib = shap_values[i]
reconstruido = base_value + contrib.sum()
pred = float(model.predict(Xte[i:i+1])[0])
print(f'base {base_value:.4f} + suma SHAP {contrib.sum():.4f} = {reconstruido:.4f}')
print(f'prediccion del modelo                          = {pred:.4f}')
assert abs(reconstruido - pred) < 1e-3, 'SHAP debe reconstruir la prediccion (aditividad)'

o = np.argsort(np.abs(contrib))
plt.figure(figsize=(6, 4))
plt.barh(np.array(feat)[o], contrib[o],
         color=['#c33' if c < 0 else '#37a' for c in contrib[o]])
plt.axvline(0, color='k', lw=0.6); plt.title(f'Waterfall - muestra #{i}')
plt.tight_layout(); plt.show()
print('OK: propiedad de aditividad local verificada')

**Ejercicio 4 — Dependence plot.** SHAP de `MedInc` vs su valor: la pendiente no es constante (no-linealidad que un coeficiente lineal no capta).

In [ ]:
j = feat.index('MedInc')
plt.figure(figsize=(6, 4))
plt.scatter(Xte[:, j], shap_values[:, j], s=10, c=Xte[:, feat.index('AveRooms')],
            cmap='coolwarm')
plt.axhline(0, color='k', lw=0.5); plt.colorbar(label='AveRooms')
plt.xlabel('MedInc'); plt.ylabel('SHAP de MedInc')
plt.title('Dependence plot: no-linealidad'); plt.tight_layout(); plt.show()

lo = shap_values[Xte[:, j] < np.median(Xte[:, j]), j].mean()
hi = shap_values[Xte[:, j] >= np.median(Xte[:, j]), j].mean()
print(f'SHAP medio de MedInc  bajo: {lo:+.3f}  |  alto: {hi:+.3f}')
print('El efecto cambia de signo/magnitud: relacion no lineal.')

**Ejercicio 5 — Interaction values.** Con `pred_interactions` (o `shap_interaction_values`) hallamos el par de features con mayor interacción.

In [ ]:
if SHAP_OK:
    inter = np.asarray(shap.TreeExplainer(model).shap_interaction_values(Xte))
else:
    booster = model.get_booster()
    inter = booster.predict(xgb.DMatrix(Xte), pred_interactions=True)[:, :-1, :-1]

# fuerza media de interaccion (fuera de la diagonal)
M = np.abs(inter).mean(0)
np.fill_diagonal(M, 0.0)
a, b = np.unravel_index(np.argmax(M), M.shape)
print(f'par con mayor interaccion: {feat[a]} x {feat[b]}  (fuerza {M[a, b]:.4f})')
plt.figure(figsize=(5, 4))
plt.imshow(M, cmap='viridis'); plt.colorbar(label='|interaccion| media')
plt.xticks(range(len(feat)), feat, rotation=90, fontsize=7)
plt.yticks(range(len(feat)), feat, fontsize=7)
plt.title('Matriz de interacciones SHAP'); plt.tight_layout(); plt.show()